# Agent Memory Architectures

## Why this matters

The X++ review agents have effectively zero memory -- each `run(xpp)` call is stateless, no agent knows what it found last time it reviewed this same class, and nothing persists between review runs. That's the right design for that specific tool (each review should be independent, and persistence would actually undermine the reproducibility work done there). But it's worth being precise about *why* stateless was right there, because the Family Finance project is the opposite case -- it needs to remember prior conversation context, user preferences, and transaction history across sessions -- and knowing which memory architecture fits which need is the actual skill, not memorizing a list of memory types.

## Level 1: What it is

An LLM call itself is stateless -- the model has no persistent memory of anything outside what's in the current context window. "Agent memory" is the application-level engineering built around that stateless call to simulate continuity. Three commonly distinguished types, borrowed loosely from cognitive-science terminology:

1. **Working memory** -- the current context window itself: the conversation so far, retrieved documents, tool results from this session. Exists only for the duration of the current interaction/session.
2. **Episodic memory** -- records of specific past interactions/events ("last Tuesday, the user asked about X and I responded Y") that can be retrieved and referenced in future sessions, typically stored outside the model and injected back into context when relevant.
3. **Long-term / semantic memory** -- durable facts and preferences distilled *from* episodic history ("this user prefers concise answers," "this user's D365 project is called IGMS") rather than raw transcripts of what was said -- generally more compact and more directly useful for personalization than replaying entire past conversations.

## Level 2: How it actually works internally

### Working memory: just the context window, with a hard ceiling

Nothing exotic here -- it's literally the messages array sent with `client.messages.create()`, exactly like the X++ project's `messages=[{"role": "user", "content": user_content}]`. The X++ agents' entire "memory" for a review is this single call's context: the class source code, the system prompt, nothing else. This is genuinely all working memory is -- there's no separate mechanism, just tokens in the same context window that gets processed together.

### Episodic memory: retrieval, not magic persistence

Under the hood, episodic memory is usually implemented as exactly the RAG pattern already covered in this track: past interactions get stored (often summarized, sometimes with embeddings computed for semantic search over them later), and when a new session starts, a retrieval step pulls back whichever past episodes seem relevant to the current query, injecting them into the new session's context window. This is why "agent memory" and "RAG" aren't actually separate topics at the implementation level for episodic memory specifically -- it's RAG applied to a store of *past conversation history* instead of a store of *documents*. The Family Finance project's chat history, if it needed to recall "what did I ask about last month's Grab spending," would need exactly this pattern: past messages stored, retrieved by relevance to the current query, injected back in.

### Long-term/semantic memory: extraction, not just storage

This is the layer that requires an actual extraction step, not just storage-and-retrieval. Raw conversation transcripts (episodic memory) get periodically processed -- usually by an LLM call itself -- to distill durable facts: "the user prefers SGD over INR by default," "the user's two sons are approximately 3 and 1," "D365 must always be referred to as D365 FnO." This distillation step is itself an LLM summarization/extraction task, with the same structured-output considerations as everything else in this track -- you'd want a schema-enforced extraction (tool-use) rather than free-text summarization, for the same reliability reasons covered in the structured-output notebook. Once extracted, these facts are typically stored separately from raw episodic history and retrieved/injected more liberally, since they're compact -- a handful of durable facts costs far fewer tokens per session than retrieving raw past transcripts would.

## Level 3: Where it breaks, and what it doesn't solve

1. **Memory can go stale, and nothing automatically detects that.** A long-term fact extracted once ("user works at Company X") doesn't automatically update when it becomes false (user changes jobs) -- unlike a database with explicit update operations, an LLM-extracted memory store needs its own explicit mechanism for detecting and correcting outdated facts, or it silently degrades into confidently wrong personalization over time.

2. **Extraction quality is the whole ballgame, same failure mode as GraphRAG's entity extraction.** If the summarization/extraction step that builds long-term memory misreads or over-generalizes from a single conversation (extracting "user is always in a hurry" from one impatient message), that becomes a durable, incorrectly-generalized fact that colors every future interaction -- and unlike a single bad RAG chunk retrieval, a wrong personalization fact actively *shapes tone and content* in ways a user may never notice is based on a bad inference.

3. **Retrieval-based episodic memory has the exact same relevance-ranking failure modes as document RAG.** If the retrieval step for "which past episodes are relevant to this new query" is a semantic similarity search, it can miss relevant history phrased very differently, or surface irrelevant history that happens to be lexically/semantically similar -- same fragility as any RAG retrieval step, just applied to conversations instead of documents.

4. **Statelessness is sometimes the correct design, not a limitation to fix -- and the X++ project is the clean example.** Adding persistent memory to the X++ review agents ("remember what Security found last time") would actively work against the reproducibility goal that project spent so much effort achieving -- a review's output should depend only on the code being reviewed, not on some hidden state from a prior run that could make identical input produce different output for a *new*, memory-driven reason on top of the LLM-judgment variance already there. Not every agent needs memory; knowing when statelessness is the feature, not the gap, is part of the actual design skill.

5. **Privacy and scope boundaries need explicit design, not assumption.** Long-term memory that persists across sessions raises real questions about what's remembered, for how long, and whether it's ever inappropriate to surface (this environment's own memory system explicitly excludes sensitive-attribute-based personalization by default, and never applies memories that could encourage unsafe behavior, precisely because "the system technically remembered this" doesn't mean "the system should always act on it") -- any agent memory system needs equivalent explicit boundaries, not just "store everything, retrieve when relevant."

In [ ]:
# Conceptual sketch of the long-term-memory extraction step -- turning
# a raw conversation transcript into durable, compact facts via a
# schema-enforced tool call (same structured-output pattern as
# base_agent.py's ISSUE_TOOL, applied to memory extraction instead of
# code review findings).

from anthropic import Anthropic

client = Anthropic()

MEMORY_EXTRACTION_TOOL = {
    "name": "extract_durable_facts",
    "description": "Extract durable, reusable facts from a conversation -- "
                    "not a summary of what was discussed, only facts likely "
                    "to still be true and useful in future sessions.",
    "input_schema": {
        "type": "object",
        "properties": {
            "facts": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "fact": {"type": "string"},
                        "confidence": {
                            "type": "string",
                            "enum": ["high", "medium", "low"],
                            "description": "How confident the extraction is "
                                            "this is a stable, durable fact "
                                            "vs. a one-off statement.",
                        },
                    },
                    "required": ["fact", "confidence"],
                },
            },
        },
        "required": ["facts"],
    },
}

transcript_excerpt = (
    "User: I always want dual currency tracking, SGD and INR together. "
    "Also I'm swamped today, can you just give me the quick answer?"
)

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    tools=[MEMORY_EXTRACTION_TOOL],
    tool_choice={"type": "tool", "name": "extract_durable_facts"},
    messages=[{"role": "user", "content": f"Extract durable facts from:\n\n{transcript_excerpt}"}],
)

extracted = next(b for b in response.content if b.type == "tool_use").input
print(extracted)
# Expect something like: dual-currency preference at HIGH confidence
# (stated as an "always" preference), being busy TODAY at LOW confidence
# (one-off state, not a durable trait) -- a well-built extractor should
# distinguish these, exactly the point in Level 3 about over-generalizing
# from a single conversation.

## Interview-ready summary

**Q: "How would you design memory for an agent that needs to personalize across sessions?"**

Weak answer: "I'd store the conversation history and load it back in next time."

Strong answer: "I'd separate working memory -- the current context window, which is free and automatic -- from episodic memory, which is retrieval over stored past interactions, essentially RAG applied to conversation history instead of documents, and from long-term/semantic memory, which is durable facts *distilled* from episodes via an extraction step, not raw transcripts. Raw episodic storage gets expensive and noisy to retrieve from at scale; distilled long-term facts are compact and more directly useful, but the extraction step needs to be conservative about confidence -- a one-off statement shouldn't get generalized into a permanent trait, and stale facts need a way to be corrected, not just accumulated forever. And critically, not every agent should have persistent memory at all -- a system where output needs to be reproducible from input alone, like a code review agent, should stay stateless by design, since memory would undermine exactly the determinism that system needs."